# Week 6 — Integrative Data Science Capstone

## End-to-end project: Wine Quality Prediction and Segmentation

### Capstone question

**Can physicochemical measurements predict red-wine quality, and can the same measurements reveal meaningful clusters of chemically similar wines?**

This notebook integrates:
- data acquisition,
- data cleaning,
- EDA,
- supervised learning,
- unsupervised learning,
- evaluation,
- recommendations,
- and reflection.


## 1. Public dataset

**UCI Wine Quality — Red Wine**

Official page:
https://archive.ics.uci.edu/dataset/186/wine+quality

Official red-wine file:
https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv

The dataset contains physicochemical variables and an integer wine-quality score.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from capstone import (
    FEATURES, TARGET, UCI_WINE_URL,
    load_wine_csv, clean_wine_data, prepare_xy,
    eda_summary, run_eda_figures,
    build_models, regression_cv, fit_and_evaluate_models,
    feature_importance_table, plot_model_comparison, plot_feature_importance,
    kmeans_analysis, save_json, write_capstone_summary
)


## 2. Acquire the public data

The project uses the official UCI CSV. If the network is unavailable, the bundled sample fixture is used so the entire pipeline can still be demonstrated offline.


In [ ]:
raw_path = ROOT / "data" / "raw" / "winequality-red.csv"
sample_path = ROOT / "data" / "sample" / "winequality-red-sample.csv"

try:
    from capstone import download_wine_quality
    if not raw_path.exists():
        download_wine_quality(raw_path)
    raw_df = load_wine_csv(raw_path)
    acquisition_mode = "official UCI"
except Exception as exc:
    print("Official acquisition unavailable:", exc)
    raw_df = load_wine_csv(sample_path)
    acquisition_mode = "sample fixture"

print("Acquisition mode:", acquisition_mode)
print("Raw shape:", raw_df.shape)
raw_df.head()


## 3. Clean the data

Cleaning rules:
- standardize column names,
- convert values to numeric,
- remove duplicates,
- remove rows with missing values,
- validate plausible physical ranges,
- preserve a quality report.


In [ ]:
cleaned, quality_report = clean_wine_data(raw_df)
print(quality_report)
print("Cleaned shape:", cleaned.shape)
cleaned.head()


## 4. Exploratory Data Analysis

We examine:
- the target distribution,
- all physicochemical feature distributions,
- feature correlations,
- and alcohol by quality group.

These analyses help identify structure before modeling.


In [ ]:
eda_summary(cleaned).head(15)


In [ ]:
EDA_DIR = ROOT / "outputs" / "figures"
run_eda_figures(cleaned, EDA_DIR)
print("EDA figures created.")


### Interpretation

The EDA should be used to:
- identify skewed features,
- note correlations among chemical variables,
- understand how the quality label is distributed,
- and formulate hypotheses about which features may help prediction.

EDA is descriptive; it does not establish that a feature causes wine quality.


## 5. Supervised learning — regression

### Problem
Predict `quality` as a continuous target.

### Models
- Dummy mean baseline
- Ridge regression
- Random Forest regressor
- Gradient Boosting regressor

### Validation
5-fold cross-validation using:
- MAE
- RMSE
- R²

The model with the lowest cross-validation RMSE is treated as the leading predictive model.


In [ ]:
X, y = prepare_xy(cleaned)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
models = build_models()

cv_results = regression_cv(models, X_train, y_train, cv)
cv_results


### Metric interpretation

- **MAE:** average absolute prediction error in quality-score units.
- **RMSE:** penalizes larger errors more strongly than MAE.
- **R²:** proportion of variance explained relative to a mean baseline.

No single metric is sufficient; all three should be viewed together.


In [ ]:
test_results, fitted_models = fit_and_evaluate_models(
    models,
    X_train,
    y_train,
    X_test,
    y_test,
)
test_results


In [ ]:
plot_model_comparison(
    test_results,
    ROOT / "outputs" / "figures" / "05_model_comparison.png",
)


## 6. Feature importance and interpretation

For tree-based models, permutation importance on held-out test data gives a model-agnostic view of which original features most affect predictive performance.


In [ ]:
best_model_name = cv_results.iloc[0]["model"]
best_model = fitted_models[best_model_name]

importance_model = best_model
if best_model_name not in {"Random Forest", "Gradient Boosting"}:
    importance_model = fitted_models["Random Forest"]

importance = feature_importance_table(
    importance_model,
    X_test,
    y_test,
)

importance.head(10)


In [ ]:
plot_feature_importance(
    importance,
    ROOT / "outputs" / "figures" / "06_feature_importance.png",
)


### Interpretation

Feature importance is not the same as causality. Strongly correlated chemistry variables can share predictive information, so importance should be treated as evidence of predictive usefulness rather than a proof that a variable directly determines quality.


## 7. Unsupervised learning — K-Means

The clustering stage intentionally excludes `quality`.

Only the physicochemical measurements are standardized and clustered, allowing the algorithm to discover chemically similar wine groups independently of the quality target.


In [ ]:
k_metrics, best_k, kmeans_model, scaler, cluster_profile, cluster_sizes = kmeans_analysis(
    X,
    ROOT / "outputs" / "figures",
)

print("Selected k:", best_k)
k_metrics


In [ ]:
print("Cluster sizes:")
cluster_sizes


### Cluster interpretation

The cluster-profile heatmap shows the mean standardized feature values for each segment.

A positive value means that a cluster is above the overall standardized mean for that feature; a negative value means it is below.

The appropriate interpretation is **similarity-based segmentation**, not a claim that a cluster is a true biological or business category.


## 8. Integrated findings

The capstone connects the two modeling perspectives:

### Predictive view
Which chemical variables are most useful for estimating quality?

### Descriptive view
Which wines are chemically similar to each other?

Together, these questions give a richer picture than using either supervised or unsupervised learning alone.


In [ ]:
prediction = fitted_models[best_model_name].predict(X_test)

model_results = pd.DataFrame(
    {
        "actual_quality": y_test.to_numpy(),
        "predicted_quality": prediction,
    }
)
model_results["absolute_error"] = (
    model_results["actual_quality"] - model_results["predicted_quality"]
).abs()

print("Best model:", best_model_name)
print("Mean absolute test error:", model_results["absolute_error"].mean())
print("Selected cluster count:", best_k)


## 9. Recommendations

1. Use the best validated regression model as a **decision-support tool**, not as the sole quality decision.
2. Focus future data collection and feature-engineering work on the variables that remain strongly predictive across validation folds.
3. Use the K-Means segments for exploratory profiling, sensory comparison, or production-batch analysis.
4. Validate the pipeline on future production data before deployment.
5. Monitor model performance over time because supplier mix, production conditions, and scoring practices can change.


## 10. Challenges and reflection

### Challenge 1 — Data quality
Real-world CSV files may contain inconsistent formatting, duplicates, missing observations, or impossible values. The pipeline handles these systematically and records the cleaning impact.

### Challenge 2 — Model selection
A single train/test split can be unstable. Cross-validation provides a stronger basis for selecting the leading regression model.

### Challenge 3 — Unsupervised interpretation
Clusters do not come with labels. Their meaning must be inferred from feature profiles and validated with domain knowledge.

### Challenge 4 — Correlation and importance
Physicochemical features may be correlated, so importance rankings should not be treated as independent causal effects.

### Successes
- A reproducible end-to-end workflow.
- Clear separation between training and test data.
- Both predictive and descriptive modeling.
- Automated testing and saved artifacts.


## 11. Limitations and next steps

### Limitations
- Wine quality scores are ordinal and can contain evaluator variability.
- The UCI dataset is relatively small for modern machine learning.
- K-Means assumes compact distance-based clusters.
- External validation on new wine batches is still required.

### Further analysis
- Try ordinal classification in addition to regression.
- Add calibrated uncertainty estimates.
- Compare gradient boosting libraries.
- Analyze cluster stability across random seeds.
- Test models on future or external wine-quality datasets.


## 12. Conclusion

This project demonstrates the full data-science lifecycle in Python:

**Acquire → Clean → Explore → Model → Evaluate → Interpret → Recommend**

It integrates supervised regression and unsupervised clustering in one coherent capstone and produces reusable code, figures, tables, models, and documentation suitable for GitHub submission.
